# Log-LOS Gradient Boosting Model

This notebook trains a nonlinear tabular model for ICU length of stay using `log1p(duration)` as the target.

Because `event_observed` is currently all `1`, this is a direct LOS prediction model rather than a censored survival model. It is intended as the next model above the log-normal AFT baseline from notebook 12.

## Imports

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

import numpy as np
import pandas as pd
from lifelines.utils import concordance_index
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, median_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from config import PROCESSED_DATA_DIR

## Load Data

In [2]:
modeling_df = pd.read_parquet(PROCESSED_DATA_DIR / "modeling_dataset.parquet")
ready_predictor_cols = pd.read_csv(
    PROCESSED_DATA_DIR / "modeling_ready_predictor_columns.csv"
)["predictor_column"].tolist()
ready_numeric_cols = pd.read_csv(
    PROCESSED_DATA_DIR / "modeling_ready_numeric_columns.csv"
)["numeric_predictor_column"].tolist()
ready_categorical_cols = pd.read_csv(
    PROCESSED_DATA_DIR / "modeling_ready_categorical_columns.csv"
)["categorical_predictor_column"].tolist()
split_df = pd.read_csv(PROCESSED_DATA_DIR / "modeling_train_test_split.csv")

print(f"Modeling dataset: {modeling_df.shape}")
print(f"Ready predictors: {len(ready_predictor_cols)}")
print(split_df["split"].value_counts())

Modeling dataset: (94444, 457)
Ready predictors: 443
split
train    75380
test     19064
Name: count, dtype: int64


In [3]:
count_cols = [col for col in ready_predictor_cols if col.endswith("_count_24h")]
model_predictor_cols = [col for col in ready_predictor_cols if col not in count_cols]

numeric_cols = [col for col in ready_numeric_cols if col in model_predictor_cols]
categorical_cols = [col for col in ready_categorical_cols if col in model_predictor_cols]

print(f"Dropped raw count predictors: {len(count_cols)}")
print(f"Model predictors: {len(model_predictor_cols)}")
print(f"Numeric predictors: {len(numeric_cols)}")
print(f"Categorical predictors: {len(categorical_cols)}")

Dropped raw count predictors: 48
Model predictors: 395
Numeric predictors: 383
Categorical predictors: 12


## Train/Test Split

In [4]:
model_df = modeling_df[
    ["subject_id", "stay_id", "duration", "event_observed"] + model_predictor_cols
].merge(
    split_df[["stay_id", "split"]],
    on="stay_id",
    how="inner",
)

if len(model_df) != len(modeling_df):
    raise ValueError("Split merge changed row count")

if model_df["stay_id"].duplicated().any():
    raise ValueError("Duplicate stay_id rows after split merge")

train_df = model_df[model_df["split"].eq("train")].copy()
test_df = model_df[model_df["split"].eq("test")].copy()

X_train = train_df[model_predictor_cols]
X_test = test_df[model_predictor_cols]
y_train = np.log1p(train_df["duration"])
y_test = np.log1p(test_df["duration"])

print(f"Train rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")
print(train_df["duration"].describe())

Train rows: 75,380
Test rows: 19,064
count    75380.000000
mean         3.617537
std          5.373389
min          0.001250
25%          1.095943
50%          1.962668
75%          3.843134
max        226.403079
Name: duration, dtype: float64


## Preprocessing and Model

Numeric features are median-imputed. Categorical features are filled with `Missing` and one-hot encoded. The model predicts `log1p(LOS days)`, then predictions are transformed back with `expm1`.

In [5]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_cols),
        ("categorical", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

gb_regressor = HistGradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=31,
    l2_regularization=0.1,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42,
)

gb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", gb_regressor),
    ]
)

gb_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tra

## Fit Model

In [6]:
gb_pipeline.fit(X_train, y_train)

fitted_model = gb_pipeline.named_steps["model"]
print(f"Iterations used: {fitted_model.n_iter_}")
print(f"Train score available: {hasattr(fitted_model, 'train_score_')}")

Iterations used: 300
Train score available: True


## Evaluate Predictions

In [7]:
train_pred_log = gb_pipeline.predict(X_train)
test_pred_log = gb_pipeline.predict(X_test)

train_pred_days = np.expm1(train_pred_log).clip(min=0)
test_pred_days = np.expm1(test_pred_log).clip(min=0)

train_actual_days = train_df["duration"].to_numpy()
test_actual_days = test_df["duration"].to_numpy()

metrics = {
    "model": "hist_gradient_boosting_log_los",
    "train_c_index": concordance_index(train_actual_days, train_pred_days, train_df["event_observed"]),
    "test_c_index": concordance_index(test_actual_days, test_pred_days, test_df["event_observed"]),
    "train_mae_days": mean_absolute_error(train_actual_days, train_pred_days),
    "test_mae_days": mean_absolute_error(test_actual_days, test_pred_days),
    "train_median_ae_days": median_absolute_error(train_actual_days, train_pred_days),
    "test_median_ae_days": median_absolute_error(test_actual_days, test_pred_days),
    "train_rmse_days": root_mean_squared_error(train_actual_days, train_pred_days),
    "test_rmse_days": root_mean_squared_error(test_actual_days, test_pred_days),
    "train_median_pred_days": np.median(train_pred_days),
    "test_median_pred_days": np.median(test_pred_days),
}

metrics_df = pd.DataFrame([metrics])
metrics_df

,model,train_c_index,test_c_index,train_mae_days,test_mae_days,train_median_ae_days,test_median_ae_days,train_rmse_days,test_rmse_days,train_median_pred_days,test_median_pred_days
0,hist_gradient_boosting_log_los,0.779661,0.761486,1.840963,2.038266,0.783363,0.841811,4.404514,4.901499,2.222473,2.248711


## Prediction Tail Diagnostics

In [8]:
prediction_tail_df = pd.DataFrame(
    {
        "split": ["train", "test"],
        "observed_max_days": [train_actual_days.max(), test_actual_days.max()],
        "predicted_max_days": [train_pred_days.max(), test_pred_days.max()],
        "predicted_99th_pct_days": [np.quantile(train_pred_days, 0.99), np.quantile(test_pred_days, 0.99)],
        "predicted_999th_pct_days": [np.quantile(train_pred_days, 0.999), np.quantile(test_pred_days, 0.999)],
    }
)

prediction_tail_df

,split,observed_max_days,predicted_max_days,predicted_99th_pct_days,predicted_999th_pct_days
0,train,226.403079,30.986516,10.621757,17.113391
1,test,121.303368,27.133458,9.945802,16.374322


In [9]:
test_predictions_df = test_df[
    ["subject_id", "stay_id", "duration", "event_observed"]
].copy()
test_predictions_df["predicted_los_days"] = test_pred_days
test_predictions_df["absolute_error_days"] = (
    test_predictions_df["predicted_los_days"] - test_predictions_df["duration"]
).abs()
test_predictions_df["predicted_los_percentile"] = test_predictions_df[
    "predicted_los_days"
].rank(pct=True)

test_predictions_df.sort_values("absolute_error_days", ascending=False).head(20)

,subject_id,stay_id,duration,event_observed,predicted_los_days,absolute_error_days,predicted_los_percentile
2286,10253349,35629939,121.303368,1,1.789610,119.513758,0.335501
16820,11774442,39245279,112.050741,1,7.468712,104.582028,0.965537
61717,16534814,32380519,103.499005,1,6.829281,96.669723,0.952056
41721,14411859,38018615,101.726238,1,5.872193,95.854045,0.923416
6482,10699336,31879957,99.638449,1,9.123296,90.515153,0.985470
90870,19624089,33576993,91.013762,1,8.443590,82.570172,0.980067
4133,10454216,39132344,77.282720,1,2.780926,74.501794,0.631662
46491,14923562,38606468,83.115046,1,8.629639,74.485407,0.982165
2807,10304606,36975538,75.099225,1,2.488751,72.610473,0.564624
34875,13697731,37151963,77.740706,1,7.839874,69.900832,0.971884


## LOS Groups

In [10]:
los_group_df = test_predictions_df.copy()
los_group_df["predicted_los_group"] = pd.qcut(
    los_group_df["predicted_los_days"],
    q=5,
    labels=["shortest", "short", "middle", "long", "longest"],
    duplicates="drop",
)

los_group_summary_df = (
    los_group_df
    .groupby("predicted_los_group", observed=True)
    .agg(
        rows=("stay_id", "size"),
        median_observed_los=("duration", "median"),
        mean_observed_los=("duration", "mean"),
        median_predicted_los=("predicted_los_days", "median"),
        mean_predicted_los=("predicted_los_days", "mean"),
    )
    .reset_index()
)

los_group_summary_df

,predicted_los_group,rows,median_observed_los,mean_observed_los,median_predicted_los,mean_predicted_los
0,shortest,3813,0.827407,1.094058,1.124651,1.015497
1,short,3813,1.351586,1.960630,1.702076,1.703063
2,middle,3812,1.933773,2.858824,2.248711,2.265772
3,long,3813,2.898669,4.272569,3.160616,3.198349
4,longest,3813,5.351285,8.210722,5.342600,5.983088


## Compare Against Log-Normal AFT Baseline

In [11]:
aft_metrics_path = PROCESSED_DATA_DIR / "model_outputs" / "lognormal_aft_metrics.csv"

if aft_metrics_path.exists():
    aft_metrics_df = pd.read_csv(aft_metrics_path)
    comparison_df = pd.concat(
        [
            aft_metrics_df.assign(source="lognormal_aft_baseline"),
            metrics_df.assign(source="gradient_boosting_log_los"),
        ],
        ignore_index=True,
        sort=False,
    )
    display_cols = [
        "source",
        "model",
        "train_c_index",
        "test_c_index",
        "train_mae_days",
        "test_mae_days",
        "train_median_ae_days",
        "test_median_ae_days",
        "train_rmse_days",
        "test_rmse_days",
    ]
    comparison_df[display_cols]
else:
    print("Log-normal AFT metrics not found; run notebook 12 first for comparison.")

## Save Outputs

In [12]:
model_output_dir = PROCESSED_DATA_DIR / "model_outputs"
model_output_dir.mkdir(exist_ok=True)

metrics_df.to_csv(model_output_dir / "gb_log_los_metrics.csv", index=False)
prediction_tail_df.to_csv(model_output_dir / "gb_log_los_prediction_tail.csv", index=False)
test_predictions_df.to_csv(model_output_dir / "gb_log_los_test_predictions.csv", index=False)
los_group_summary_df.to_csv(model_output_dir / "gb_log_los_group_summary.csv", index=False)
pd.Series(count_cols, name="dropped_count_predictor").to_csv(
    model_output_dir / "gb_log_los_dropped_count_predictors.csv",
    index=False,
)

print(f"Saved gradient boosting outputs to {model_output_dir}")

Saved gradient boosting outputs to /Users/brandonwu/Documents/ICU_LOS_Survival_Analysis/data/processed/model_outputs


## Modeling Notes

In [13]:
print("Modeling notes:")
print("- This model is not censored-survival; it predicts log1p(ICU LOS) directly.")
print("- It is appropriate here because event_observed is currently all 1.")
print("- Compare against notebook 12 on MAE/RMSE, C-index, and prediction-tail behavior.")
print("- Raw count features are excluded for the same charting-density reason as notebook 12.")

Modeling notes:
- This model is not censored-survival; it predicts log1p(ICU LOS) directly.
- It is appropriate here because event_observed is currently all 1.
- Compare against notebook 12 on MAE/RMSE, C-index, and prediction-tail behavior.
- Raw count features are excluded for the same charting-density reason as notebook 12.
